# <font color='#000000'>__Exercício Prático 2__</font>
## <font color='#1c8a23'>Criação de Métrica - Feeds OPTA</font>
#### <font color='#4b4b4b'>Master Big Data Aplicado ao Futebol <br> Módulo 5 - Análise de dados no futebol com Python <br> Desenvolvido por: Hugo Alves </font>

__Enunciado do Exercício__

<i> Com base nos feeds F24 que a OPTA nos disponibilizou, desafio-vos a criar uma métrica (podem até tentar quantificar aquela que inventaram no modulo 2) e a calculá-la para todos os jogadores ou equipas de uma das ligas, analisando depois os resultados.

Dependendo da métrica, poderá ser interessante calculá-la apenas para jogadores de uma determinada posição, apenas para ações realizadas numa determinada zona do campo, apenas num momento de jogo ou até apenas em determinadas condições de gamestate (resultado no marcados), por exemplo: passes falhados pelos centrais em zona defensiva nos últimos 15 minutos de cada jogo quando a equipa está em desvantagem. <br>
Pode ser uma métrica coletiva ou individual, com liberdade total para criarem com base nos dados que temos, desde que depois seja aplicada a todas as equipas/jogadores e os resultados sejam contextualizados. </i>
<br><br>
__Nota:__ Devido ao espaço ocupado pelos ficheiros XML com os dados da OPTA, este notebook foi preparado para correr em Google Colab.

# <font color="#1c8a23">___________________</font>
## <font color="#4b4b4b">Índice de Conteúdos</font> <a class="anchor" id="toc"></a>
[1. Setup Inicial](#setup)<br>
- [1.1. Packages e Funções](#pack)<br>
- [1.2. Ficheiros de Eventos](#event)<br>

[2. Definição e Cálculo das Métricas](#metricas)<br>
- [2.1. Distância Média dos Passes](#dist)<br>
- [2.2. Percentagem de Sucesso dos Passes](#success)<br>

[3. Análise dos Resultados](#analise)<br>

# <font color="#1c8a23">____________</font>
## <font color='#4b4b4b'>1. Setup Inicial</font> <a class="anchor" id="setup"></a>
[Regressar ao Índice](#toc)

### <font color='#1c8a23'>1.1. Packages e Funções</font> <a class="anchor" id="pack"></a>
[Regressar ao Índice](#toc)

In [ ]:
!python --version

Vamos utilizar a versão 3.12.12 do Python.

Abaixo encontram-se os packages utilizados ao longo deste notebook. À partida, o único que necessitará de ser instalado previamente no Colab será o package `plottable`. Depois de instalado, a célula abaixo poderá ser comentada.

In [ ]:
!pip install plottable

In [ ]:
import os
import xml.etree.ElementTree as ET
import time
from tqdm.notebook import tqdm
from typing import Optional
from google.colab import files
import zipfile

import numpy as np
import pandas as pd

import requests
from bs4 import BeautifulSoup

from plottable import Table, ColumnDefinition
from plottable.cmap import normed_cmap
import matplotlib.pyplot as plt
import matplotlib

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

Na célula seguinte encontram-se as funções utilizadas ao longo deste notebook. O ideal seria ter um ficheiro .py separado com as funções, mas neste caso, em que estamos a trabalhar num ficheiro colaborativo, talvez seja mais prático ter tudo junto.

In [ ]:
# Função auxiliar para mover colunas
def move_column(df: pd.DataFrame,
                col_to_move: str,
                after_col: str) -> pd.DataFrame:
  """
  Move uma coluna para uma posição específica de um DataFrame.

  Parâmetros:
  - df (pd.DataFrame): DataFrame a alterar.
  - col_to_move (str): Coluna a mover.
  - after_col (str): Coluna que deve anteceder a coluna movida.

  Devolve:
  - pd.DataFrame: DataFrame com a coluna movida.
  """
  cols = list(df.columns)
  cols.remove(col_to_move)
  insert_at = cols.index(after_col) + 1
  cols.insert(insert_at, col_to_move)
  return df[cols]

# Função para extrair dados de uma pasta com ficheiros XML para um DataFrame
def parse_F24_folder(folder_path: str,
                     event_descriptions: pd.DataFrame,
                     player_descriptions: Optional[pd.DataFrame] = None) -> pd.DataFrame:
  """
  Recebe uma pasta com ficheiros XML de dados da OPTA (F24) e devolve um DataFrame com os dados transformados.
  Além de trazer os dados para um DataFrame único, aplica transformações sobre as colunas, data types, e
  complementa com a descrição dos eventos e jogadores que os executaram (este último opcional).

  Parâmetros:
  - folder_path (str): Caminho para a pasta com os ficheiros XML.
  - event_descriptions (pd.DataFrame): DataFrame com a descrição dos eventos. Deve conter as colunas "type_id" e "event_name".
  - player_descriptions (pd.DataFrame): DataFrame com a descrição dos jogadores.

  Devolve:
  - pd.DataFrame: DataFrame com os dados transformados.
  """
  t0 = time.perf_counter()
  games_list = []
  events_list = []

  ### 1. Carregar ficheiros XML
  for file in tqdm(os.listdir(folder_path)):
    # Ignorar ficheiros que não XML
    if file.endswith(".xml"):
      file_path = os.path.join(folder_path, file)
      tree = ET.parse(file_path)
      games = tree.getroot()
      # Assumindo que cada ficheiro diz respeito a um jogo, vamos buscar os metadados dessa partida
      game_info = games.find("Game")

      game_id = game_info.get("id")
      game_meta = {
        "game_id": game_id,
        "season_id": game_info.get("season_id"),
        "season_name": game_info.get("season_name"),
        "competition_id": game_info.get("competition_id"),
        "competition_name": game_info.get("competition_name"),
        "matchday": game_info.get("matchday"),
        "game_date": game_info.get("game_date"),
        "home_team_id": game_info.get("home_team_id"),
        "home_team_name": game_info.get("home_team_name"),
        "home_score": game_info.get("home_score"),
        "away_team_id": game_info.get("away_team_id"),
        "away_team_name": game_info.get("away_team_name"),
        "away_score": game_info.get("away_score"),
      }
      games_list.append(game_meta)

      # Iterar sobre todos os eventos da partida
      for game in games:
        for event in game.findall("Event"):
          event_data = event.attrib.copy()
          event_data["game_id"] = game_id
          # Iterar sobre os qualifiers (identificados por "Q" nos ficheiros XML)
          event_data["qualifiers"] = [q.attrib for q in event.findall("Q")]
          events_list.append(event_data)

  print("Carregamento dos ficheiros XML concluído. A iniciar transformação do DataFrame")
  t1 = time.perf_counter()
  print(f"Duração do carregamento dos ficheiros XML: {t1 - t0:.2f}s")

  ### 2. Juntar jogos e eventos
  df_games = pd.DataFrame(games_list)
  df_events = pd.DataFrame(events_list)

  # Associar metadados dos jogos a cada evento
  df_events = pd.merge(df_events, df_games, how = "left", on = "game_id")

  ### 3. Transformar data types
  # Colunas numéricas (números inteiros)
  small_numeric_cols = ["event_id", "type_id", "period_id", "min", "sec", "team_id", "season_id", "competition_id",
                        "matchday", "home_team_id", "home_score", "away_team_id", "away_score"]
  medium_numeric_cols = ["game_id", "player_id"]
  big_numeric_cols = ["id", "version"]
  for col in small_numeric_cols:
    df_events[col] = pd.to_numeric(df_events[col], errors = "coerce").astype("Int16")
  for col in medium_numeric_cols:
    df_events[col] = pd.to_numeric(df_events[col], errors = "coerce").astype("Int32")
  for col in big_numeric_cols:
    df_events[col] = pd.to_numeric(df_events[col], errors = "coerce").astype("Int64")

  # Colunas numéricas (coordenadas com casas decimais)
  df_events["x"] = pd.to_numeric(df_events["x"], errors = "coerce").astype("Float32")
  df_events["y"] = pd.to_numeric(df_events["y"], errors = "coerce").astype("Float32")

  # Colunas booleanas (0 ou 1s) - nestes casos, será seguro assumir que a ausência de valor representa 0
  boolean_cols = ["outcome", "keypass", "assist"]
  for col in boolean_cols:
    # vamos utilizar o data type numérico em vez de boolean
    df_events[col] = pd.to_numeric(df_events[col], errors = "coerce").fillna(0).astype("Int8")

  # Datas
  df_events["timestamp"] = pd.to_datetime(df_events["timestamp"], errors = "coerce")
  df_events["last_modified"] = pd.to_datetime(df_events["last_modified"], errors = "coerce")
  df_events["game_date"] = pd.to_datetime(df_events["game_date"], errors = "coerce")

  # Colunas categóricas (colunas com relativamente poucos valores únicos face ao total de linhas)
  category_cols = ["season_name", "competition_name", "home_team_name", "away_team_name"]
  for col in category_cols:
    df_events[col] = df_events[col].astype("category")

  ### 4. Juntar descrição dos eventos, qualifiers e jogadores
  try:
    df_events = pd.merge(df_events, event_descriptions, how = "left", on = "type_id")
    # Mover a descrição para depois do tipo de evento
    df_events = move_column(df_events, "event_name", "type_id")
  except Exception:
    print("event_descriptions deve ser um DataFrame e conter as colunas 'type_id' e 'event_name'.")
    return

  if player_descriptions is not None:
    if isinstance(player_descriptions, pd.DataFrame) and "player_id" in player_descriptions.columns:
      # Guardar nome das colunas para adicionar prefixo às colunas novas (a função pd.merge só aceita sufixos)
      existing_cols = set(df_events.columns)
      df_events = pd.merge(df_events, player_descriptions, how = "left", on = "player_id")
      # Adicionar prefixo e mover as colunas para imediatamente depois do player_id
      new_cols = list(set(df_events.columns) - existing_cols - {"player_id"})
      new_cols_renamed = [f"player_{col}" for col in new_cols]
      df_events.rename(columns = dict(zip(new_cols, new_cols_renamed)), inplace = True)
      for col in new_cols_renamed:
        df_events = move_column(df_events, col, "player_id")
    else:
      print("player_descriptions deve ser um DataFrame e conter a coluna 'player_id'.")

  t2 = time.perf_counter()
  print("=====================================================================================")
  print(f"Transformação do DataFrame concluída em {t2 - t1:.2f}s")
  print(f"Duração total: {t2 - t0:.2f}s\n")
  return df_events

# "Explode" os eventos de um DataFrame
def explode_events(df_events: pd.DataFrame,
                   event_type: int,
                   qualifier_descriptions: Optional[pd.DataFrame] = None) -> pd.DataFrame:
  """
  "Explode" os eventos de um DataFrame. Recebe um DataFrame em que cada linha corresponde a um evento,
  e devolve um DataFrame em que cada linha passa a ser uma combinação evento & qualifier.

  Parâmetros:
  - df_events (pd.DataFrame): DataFrame com os eventos.
  - event_type (int): ID do evento a "explodir".
  - qualifier_descriptions (pd.DataFrame): DataFrame com a descrição dos qualifiers.
  """
  print("A iniciar transformação")
  t0 = time.perf_counter()

  # Criar cópia do DataFrame e validar a existência de eventos
  df = df_events[df_events["type_id"] == event_type].copy()
  if df.empty:
    print(f"Não foram encontrados eventos com o ID {event_type}.")
    return pd.DataFrame()

  # "Explodir" qualifiers (partindo do princípio que é uma lista de dicionários)
  df_exploded = df.explode("qualifiers")

  # Normalizar qualifiers
  df_q = pd.json_normalize(df_exploded["qualifiers"]).fillna("Yes")
  df_q["id"] = df_exploded["id"].values

  # Fazer pivot dos qualifiers. Para cada ID, cada qualifier passa a ser uma coluna (com o respetivo valor)
  df_q = df_q.pivot_table(index = "id", columns = "qualifier_id", values = "value", aggfunc = "first").reset_index()

  # Associar descrição dos qualifiers (se o parâmetro tiver sido passado)
  if qualifier_descriptions is not None:
    if isinstance(qualifier_descriptions, pd.DataFrame) and {"qualifier_id", "description"}.issubset(qualifier_descriptions.columns):
      dict_q = dict(zip(qualifier_descriptions["qualifier_id"].astype(str), qualifier_descriptions["description"].astype(str)))
      df_q.rename(columns = dict_q, inplace = True)
    else:
      print("qualifier_descriptions deve ser um DataFrame e conter as colunas 'qualifier_id' e 'description'.\nA continuar transformação sem a descrição dos qualifiers.")

  # Voltar a juntar os DataFrames e preencher qualifiers em falta com "-"
  qualifier_cols = df_q.columns.difference(["id"])
  df = df.drop(columns = ["qualifiers"]).merge(df_q, on = "id", how = "left")
  df[qualifier_cols] = df[qualifier_cols].fillna("-")

  t1 = time.perf_counter()
  print("=====================================================================================")
  print(f"Duração da transformação dos eventos: {t1 - t0:.2f}s\n")

  return df

### <font color='#1c8a23'>1.2. Ficheiros de Eventos</font> <a class="anchor" id="event"></a>
[Regressar ao Índice](#toc)

Vamos começar por ir buscar a pasta ZIP com os dados providenciados pela OPTA.

In [ ]:
folder_name = "OPTA Data"

In [ ]:
if os.path.isdir(folder_name):
  print(f"A pasta '{folder_name}' já existe. A usar os ficheiros existentes.")
else:
  print(f"Pasta não encontrada. Necessário fazer upload do ficheiro ZIP.")

  uploaded = files.upload()

  zip_name = list(uploaded.keys())[0]
  with zipfile.ZipFile(zip_name, "r") as zip_ref:
      zip_ref.extractall(folder_name)

In [ ]:
folder_path = "OPTA Data/OPTA Data/F24 - Portugal"

Vamos agora importar dois ficheiros auxiliares com as descrições dos IDs dos eventos e qualifiers. Estes ficheiros foram criados tendo por base os dicionários do ficheiro `parse_24` fornecido junto com o enunciado, portanto fica o agradecimento pelo trabalho poupado.

Por sua vez, este dicionário terá por base a documentação da OPTA, partilhada via PDF e disponível [online](https://github.com/jokecamp/FootballData/blob/master/random_docs/Opta-f24_appendices.docx).

In [ ]:
df_event_types = pd.read_csv("OPTA Data/OPTA Data/Event Types.csv")
df_qualifiers = pd.read_csv("OPTA Data/OPTA Data/Qualifier Descriptions.csv")
df_players = pd.read_excel("OPTA Data/OPTA Data/opta_planteis_portugal.xlsx")

De seguida, chamamos a função para extrair os dados dos ficheiros XML fornecidos. Esta função é também adaptada do ficheiro `parse_24`, embora com algumas diferenças (especialmente no que respeita ao processamento do DataFrame dos eventos).

In [ ]:
df_events = parse_F24_folder(folder_path,
                             event_descriptions = df_event_types,
                             player_descriptions = df_players)

In [ ]:
df_events.info()

In [ ]:
df_events.head()

Perfeito. Agora que já temos um dataset completo e transformado com os eventos da partida e informação complementar, podemos prosseguir para a definição das nossas métricas.

# <font color="#1c8a23">______________________________</font>
## <font color='#4b4b4b'>2. Definição e Cálculo das Métricas</font> <a class="anchor" id="metricas"></a>
[Regressar ao Índice](#toc)

Neste trabalho, __a nossa análise vai incidir sobre o comportamento das equipas ao nível do passe quando se encontram em desvantagem no marcador__, comparativamente à sua abordagem "normal" nas partidas. Cada vez mais é mencionada pelos treinadores a necessidade de manter a mesma abordagem ao jogo quer a equipa esteja ou não em desvantagem, mas é muitas vezes notória a diferença naquilo que é a postura das equipas em campo, e até no que os próprios treinadores pedem. Isso será forçosamente verdade, mais não seja porque as equipas não jogam sozinhas e os adversários também se protegem mais quando estão na frente do marcador, com o objetivo de segurar a vantagem. <br>
Para materializar esta ideia em números, iremos calcular duas métricas complementares ao nível coletivo:
* __Distância média dos passes quando em desvantagem__
* __Percentagem de passes bem sucedidos quando em desvantagem__

Estes indicadores serão calculados __apenas para os últimos 20 minutos das partidas__ (em que o "desespero" das equipas é tendencialmente superior), e terão como base de comparação (além dos valores da mesma métrica para as restantes equipas), os mesmos indicadores calculados independentemente da fase do jogo e do resultado.

Antes de avançarmos para o cálculo de cada métrica individualmente, podemos aproveitar desde já para criar variáveis auxiliares que serão cruciais na nossa análise, e que permitirão saber se a equipa se encontrava ou não em desvantagem à data do evento. A melhor forma de o fazer será contar, para um dado jogo, os eventos de golos que tinham ocorrido para cada equipa até esse momento.

In [ ]:
# Começamos por ordenar o dataset por jogo, parte, minuto, segundo, timestamp, e ID do evento (no limite para desempatar)
df_events = df_events.sort_values(["game_id", "period_id", "min", "sec", "timestamp", "id"]).reset_index(drop = True)

In [ ]:
# Variáveis auxiliares para identificar golos das equipas (ID 16, segundo o apêndice da OPTA)
df_events["home_goal"] = ((df_events["type_id"] == 16) & (df_events["team_id"] == df_events["home_team_id"])).astype("Int8")
df_events["away_goal"] = ((df_events["type_id"] == 16) & (df_events["team_id"] == df_events["away_team_id"])).astype("Int8")

In [ ]:
# Somar cumulativamente os eventos de golo de cada equipa dentro de uma partida
df_events["home_score_current"] = df_events.groupby("game_id")["home_goal"].cumsum()
df_events["away_score_current"] = df_events.groupby("game_id")["away_goal"].cumsum()

# Um check rápido para ver se parece tudo bem
df_events[df_events["type_id"] == 16]

In [ ]:
# Variável para identificar se a equipa está a perder
df_events["is_losing"] = (
    ((df_events["team_id"] == df_events["home_team_id"]) & (df_events["home_score_current"] < df_events["away_score_current"])) \
    | \
    ((df_events["team_id"] == df_events["away_team_id"]) & (df_events["away_score_current"] < df_events["home_score_current"]))
).astype("Int8")
df_events.tail()

In [ ]:
# Apagar colunas auxiliares que temos a certeza que não serão mais utilizadas
df_events.drop(columns = ["home_goal", "away_goal"], inplace = True)

Maravilha. Vamos então avançar para o cálculo das métricas, começando pela distância média dos passes das equipas quando em desvantagem nas segundas partes.

### <font color='#1c8a23'>2.1. Distância Média dos Passes</font> <a class="anchor" id="dist"></a>
[Regressar ao Índice](#toc)

Para obter esta métrica, apenas nos interessam os eventos de passes. Por isso, vamos usar a função `explode_events` para converter os qualifiers no formato tabular com que estamos a trabalhar.

__Notas:__
1) Suportando-nos do Apêndice 3 dos ficheiros F24 da OPTA ("useful queries") vamos eliminar os passes com os qualifiers 2, 5, 6, 107, 123, e 124, que não se inserem na quantificação dos passes completados.*
2) Para este indicador, vamos optar por incluir todos os passes (independentemente do seu sucesso), visto que os passes longos terão provavelmente maior probabilidade de insucesso e, se os eliminarmos, estamos a perder uma componente que pode ser relevante na nossa análise.

*Também segundo a indicação da OPTA no mesmo apêndice, deverão ser incluídos os cantos curtos, em que o qualifier 6 está presente mas não o qualifier 2. Na prática, se o qualifier 2 estiver presente o evento já será descartado de qualquer forma, pelo que podemos ignorar a condição de eliminar o qualifier 6.

In [ ]:
# ID 1, segundo o apêndice da OPTA
df_passes = explode_events(df_events, 1, df_qualifiers)
df_passes.info()

In [ ]:
df_passes.head()

Agora que temos um DataFrame com todos os passes, vamos espreitar a tabela com a descrição dos qualifiers para ver de que colunas se tratam.

In [ ]:
df_qualifiers[df_qualifiers["qualifier_id"].isin([2, 5, 107, 123, 124])]

In [ ]:
# Mostrar os valores únicos das colunas para garantir que não há surpresas
for col in ["Cross", "Free kick taken", "Throw-in", "Keeper Throw", "Goal Kick"]:
  print(f"Valores únicos da coluna {col}:")
  print(df_passes[col].unique())

Estando tudo alinhado com o que esperávamos encontrar, podemos agora eliminar os registos em que alguma destas colunas tenha o valor "Yes".

In [ ]:
passes_to_delete = (
  (df_passes["Free kick taken"] == "Yes") |
  (df_passes["Throw-in"] == "Yes") |
  (df_passes["Keeper Throw"] == "Yes") |
  (df_passes["Goal Kick"] == "Yes") |
  (df_passes["Cross"] == "Yes")
)
df_passes = df_passes[~passes_to_delete]

# Confirmar que já não temos eventos onde estes qualifiers estejam presentes
for col in ["Cross", "Free kick taken", "Throw-in", "Keeper Throw", "Goal Kick"]:
  print(f"Valores únicos da coluna {col}:")
  print(df_passes[col].unique())

Com os passes filtrados para estarem de acordo com as definições da OPTA, podemos avançar para o cálculo da métrica.

Neste caso, vamos trabalhar com a coluna `length`. Vamos convertê-la para um formato numérico tentar perceber se os valores estão todos dentro do normal.

In [ ]:
df_passes["Length"] = pd.to_numeric(df_passes["Length"], errors = "coerce")
df_passes["Length"].describe()

In [ ]:
df_passes["Length"].isna().sum()

Não há valores em falta, o que é um bom sinal. Ainda assim, olhando para a descrição deste qualifier (ID 212) nos apêndices, é indicado que este valor está em jardas e não em metros. Vamos converter para a unidade com que costumamos trabalhar (e aproveitar e ver casos de passes a maior distância para validar).

In [ ]:
# Segundo a internet, 1 jarda = 0.9144 metros
df_passes["Length"] = df_passes["Length"] * 0.9144
# É mais provável (e aceitável) que os guarda-redes façam passes a maior distâncias
df_passes[(df_passes["Length"] > 80) & (df_passes["player_position"] != "Goalkeeper")]

Estes passes (todos eles falhados) foram todos feitos por defesas. Tendo em conta que estes são, ainda assim, valores razoáveis, vamos prosseguir sem fazer nada a este respeito e avançar para o cálculo da métrica, começando por obter o nome da equipa que efetuou o passe.

In [ ]:
df_passes["team_name"] = np.where(
  df_passes["team_id"] == df_passes["home_team_id"],
  df_passes["home_team_name"],
  df_passes["away_team_name"]
)

Podemos agora criar dois DataFrames, um com a distância média dos passes (independentemente do resultado e fase do jogo) e outro com a distância média dos passes quando em desvantagem nos últimos 20 minutos.

In [ ]:
# DataFrame com valores base
df_passes_avg_dist_base = (
  df_passes
  .groupby(["team_id", "team_name"], as_index = False)
  .aggregate(avg_pass_length_base = ("Length", "mean"))
)

# DataFrame com valores para equipas em desvantagem nos últimos 20 minutos (depois dos 70')
df_passes_avg_dist_losing = (
  df_passes[(df_passes["is_losing"] == 1) & (df_passes["min"] > 70)]
  .groupby(["team_id", "team_name"], as_index = False)
  .aggregate(avg_pass_length_losing = ("Length", "mean"))
)

Por fim, podemos juntar as duas tabelas numa só, e juntar uma terceira variável para medir o desvio (sob a forma de diferença) entre os dois indicadores calculados acima.

In [ ]:
df_passes_avg_dist = df_passes_avg_dist_base.merge(
  df_passes_avg_dist_losing,
  on = ["team_id", "team_name"],
  how = "left"
)
df_passes_avg_dist["delta_pass_length"] = (df_passes_avg_dist["avg_pass_length_losing"] - df_passes_avg_dist["avg_pass_length_base"])
df_passes_avg_dist

Vamos guardar a análise dos resultados para uma fase posterior do trabalho e avançar para o cálculo da segunda métrica.

### <font color='#1c8a23'>2.2. Percentagem de Sucesso dos Passes</font> <a class="anchor" id="success"></a>
[Regressar ao Índice](#toc)

Se há pouco usámos o qualifier `Length` para medir a distância dos passes, vamos agora recorrer ao `outcome` do evento para avaliar o sucesso destas ações, e como estas variam para cada equipa quando estas se encontram em desvantagem nos últimos 20 minutos das partidas. Aqui, importa realçar que apenas vamos trabalhar com passes "legais" e, por isso, não serão considerados os passes que resultam em fora de jogo, com ID 2 de tipo de evento.

In [ ]:
# Código auxiliar para confirmar que os passes para posição de fora de jogo não são simultaneamente contabilizados como passes "legais"
# Aqui, filtramos a tabela de eventos pelo ID 2 (correspondente ao "Offside Pass") e vemos os registos imediatamente antes e depois, respetivamente
"""
pass_offside = df_events["type_id"] == 2
display(df_events[pass_offside | pass_offside.shift(-1, fill_value=False)].head(10))
print("\n\n===================================================================\n\n")
display(df_events[pass_offside | pass_offside.shift(1, fill_value=False)].head(10))
"""

In [ ]:
# Confirmar que não temos missing values escondidos nesta coluna
df_passes["outcome"].value_counts(dropna = False)

Vamos agora seguir a mesma lógica que aplicámos para a métrica anterior e criar dois DataFrames com os valores base e os valores para as equipas em desvantagem depois dos 70 minutos, que depois juntarmos num único. <br>
Neste caso, como a coluna do outcome é boolean (0 ou 1), a média da coluna dar-nos-á a percentagem de passes completados.

In [ ]:
# DataFrame com valores base
df_passes_completed_base = (
  df_passes
  .groupby(["team_id", "team_name"], as_index = False)
  .aggregate(pct_pass_completed_base = ("outcome", "mean"))
)

# DataFrame com valores para equipas em desvantagem nos últimos 20 minutos (depois dos 70')
df_passes_completed_losing = (
  df_passes[(df_passes["is_losing"] == 1) & (df_passes["min"] > 70)]
  .groupby(["team_id", "team_name"], as_index = False)
  .aggregate(pct_pass_completed_losing = ("outcome", "mean"))
)

In [ ]:
df_passes_completed = df_passes_completed_base.merge(
  df_passes_completed_losing,
  on = ["team_id", "team_name"],
  how = "left"
)
df_passes_completed["delta_pass_completed"] = (df_passes_completed["pct_pass_completed_losing"] - df_passes_completed["pct_pass_completed_base"])
df_passes_completed

Perfeito, podemos prosseguir para a análise dos resultados.

# <font color="#1c8a23">______________________</font>
## <font color='#4b4b4b'>3. Análise dos Resultados</font> <a class="anchor" id="analise"></a>
[Regressar ao Índice](#toc)

Antes de nos debruçarmos sobre as métricas, pode ser interessante obter a classificação final do campeonato para depois comparar os valores com a posição no fim da liga. Podemos usar código semelhante ao do Exercício Prático 1 para obter a tabela classificativa.

__Nota:__ Poderíamos tentar recorrer à API [football-data.org](https://www.football-data.org/), que também utilizámos para esse exercício, para obter a tabela. Contudo, os dados desta época para o campeonato português não são disponibilizados de forma gratuita, pelo que não temos como lá chegar.

In [ ]:
# Antes de mais, precisamos que cada linha corresponda a uma partida
df_matches = (
  df_events[["game_id", "home_team_id", "home_team_name", "away_team_id", "away_team_name", "home_score", "away_score"]]
  .drop_duplicates()
  .reset_index(drop = True)
)

# Fazer um check que não há jogos repetidos (que teriam informação diferente nas restantes colunas)
df_matches["game_id"].value_counts()

Não temos jogos repetidos, mas na linha acima conseguimos aperceber-nos que apenas temos 300 jogos neste novo dataset. Vamos confirmar se também só temos este número de jogos na tabela dos eventos.

In [ ]:
len(df_events["game_id"].unique().tolist())

Confere. Isto corta-nos um pouco as pernas no objetivo de chegar a uma tabela final consolidada, uma vez que há jogos em falta que poderão beneficiar algumas equipas e prejudicar outras (visto que as métricas são médias dos valores observados por jogo, mas a tabela classificativa é cumulativa).

Vamos tentar recorrer ao Transfermarkt para obter a tabela, fazendo também uso do PDF tutorial de web scraping fornecido neste módulo.

In [ ]:
# User genérico
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

# Aceder à página
page = "https://www.transfermarkt.pt/liga-portugal/startseite/wettbewerb/PO1/plus/?saison_id=2020"
pageTree = requests.get(page, headers = headers)
pageSoup = BeautifulSoup(pageTree.content, "html.parser")

In [ ]:
# Existem várias tabelas nesta página. Para encontrar a que queremos, vamos procurar por aquela que tem as colunas que precisamos
# Nota: Esta parte foi escrita com ajuda do ChatGPT
tables = pageSoup.find_all("table", class_="items")

for table in tables:
  headers = [th.text.strip() for th in table.find_all("th")]
  # Nomes das colunas com base no apresentado na página
  if {"#", "Clube", "+/-", "Pts"}.issubset(headers):
    table_standings = table
    break
else:
  raise ValueError("Tabela não encontrada")

In [ ]:
# Vamos diretamente às colunas que nos interessam
columns_standings = ["#", "Clube", "+/-", "Pts"]
rows_data = []

# Iteramos sobre todas as linhas
for row in table_standings.find("tbody").find_all("tr"):
  # Juntamos os valores relevantes à lista que depois vai alimentar o DataFrame
  cols = row.find_all("td")
  rows_data.append([
    cols[0].text.strip(),  # posição
    cols[2].text.strip(),  # clube
    cols[4].text.strip(),  # diferença de golos
    cols[5].text.strip()   # pontos
  ])

# E criamos a tabela final
df_standings = pd.DataFrame(rows_data, columns = columns_standings)
df_standings[["#", "+/-", "Pts"]] = df_standings[["#", "+/-", "Pts"]].astype(int)
df_standings.head()

Ótimo. Ainda assim, surge um novo problema: associar as equipas entre a fonte da OPTA e os dados do Transfermarkt (visto que os IDs não são os mesmos e o nome das equipas também surge escrito de forma diferente). Vamos criar um dicionário para mapear manualmente os nomes do Transfermarkt para bater certo com os da OPTA, e depois podemos então juntar as três tabelas numa só.

In [ ]:
team_names_transfermarkt_to_opta = {
  "Sporting": "Sporting CP",
  "FC Porto": "FC Porto",
  "Benfica": "Benfica",
  "Braga": "Sporting Braga",
  "P. Ferreira": "Paços de Ferreira",
  "Santa Clara": "Santa Clara",
  "Vitória SC": "Vitória Guimarães",
  "Moreirense": "Moreirense",
  "Famalicão": "FC Famalicão",
  "B SAD": "Belenenses",
  "Gil Vicente": "Gil Vicente",
  "Tondela": "Tondela",
  "Boavista": "Boavista",
  "Portimonense": "Portimonense",
  "Marítimo": "Marítimo",
  "Rio Ave": "Rio Ave",
  "Farense": "SC Farense",
  "CD Nacional": "CD Nacional"
}
df_standings["team_name"] = df_standings["Clube"].map(team_names_transfermarkt_to_opta)

In [ ]:
df_passes_metrics = pd.merge(df_passes_avg_dist, df_passes_completed, on = ["team_id", "team_name"], how = "left")
df_passes_metrics = pd.merge(df_passes_metrics, df_standings, on = "team_name", how = "left")
df_passes_metrics = df_passes_metrics.sort_values("#").reset_index(drop = True)
df_passes_metrics

Finalmente temos todos os dados que precisamos. Vamos criar uma tabela com melhor aspeto e apenas com as variáveis relevantes para depois analisar (finalmente) os resultados.

In [ ]:
df_passes_metrics_visual = df_passes_metrics.copy()

cols_to_round = ["avg_pass_length_base", "avg_pass_length_losing", "delta_pass_length", "pct_pass_completed_base", "pct_pass_completed_losing", "delta_pass_completed"]
for col in cols_to_round:
  df_passes_metrics_visual[col] = df_passes_metrics_visual[col].round(2)

In [ ]:
row_colors = {"even": "#F3F3F3", "odd": "#FFFFFF"}
bg_color = row_colors["odd"]

In [ ]:
col_defs = [
  # Colunas relativas à classificação
  ColumnDefinition(name = "#", title = "", textprops = {"ha": "center", "weight": "bold"}, width = 0.5),
  ColumnDefinition(name = "team_name", title = "", textprops = {"ha": "left", "weight": "bold"}, width = 1.5),
  ColumnDefinition(name = "+/-", title = "DG", textprops = {"ha": "center"}, width = 0.5, border = "left"),
  ColumnDefinition(name = "Pts", title = "P", textprops = {"ha": "center"}, width = 0.5),
  # Colunas relativas à distância média do passe
  ColumnDefinition(name = "avg_pass_length_base", title = "Total", textprops = {"ha": "center"}, group = "Distância Média Passe (m)", width = 0.5, border = "left"),
  ColumnDefinition(name = "avg_pass_length_losing", title = "D>70'", textprops = {"ha": "center"}, group = "Distância Média Passe (m)", width = 0.5),
  ColumnDefinition(name = "delta_pass_length", title = "Δ", textprops = {"ha": "center", "bbox": {"boxstyle": "circle,pad=0.45"}},
                   group = "Distância Média Passe (m)", cmap = normed_cmap(df_passes_metrics["delta_pass_length"], cmap = matplotlib.colormaps["RdBu"]),
                   width = 0.5),
  # Colunas relativas à percentagem de passes completados
  ColumnDefinition(name = "pct_pass_completed_base", title = "Total", textprops = {"ha": "center"}, group = "Passes Completados (%)", width = 0.5, border = "left"),
  ColumnDefinition(name = "pct_pass_completed_losing", title = "D>70'", textprops = {"ha": "center"}, group = "Passes Completados (%)", width = 0.5),
  ColumnDefinition(name = "delta_pass_completed", title = "Δ", textprops = {"ha": "center", "bbox": {"boxstyle": "circle,pad=0.45"}},
                   group = "Passes Completados (%)", cmap = normed_cmap(df_passes_metrics["delta_pass_completed"], cmap = matplotlib.colormaps["RdBu"]),
                   width = 0.5)
]

In [ ]:
fig, ax = plt.subplots(figsize = (16, 18))
ax.set_axis_off()
ax.set_position([0.03, 0.07, 0.94, 0.85])
fig.set_facecolor(bg_color)
ax.set_facecolor(bg_color)

df_passes_metrics_columns = [
  "team_name", "+/-", "Pts",
  "avg_pass_length_base", "avg_pass_length_losing", "delta_pass_length",
  "pct_pass_completed_base", "pct_pass_completed_losing", "delta_pass_completed"
]

table = Table(
  df_passes_metrics_visual,
  column_definitions = col_defs,
  row_dividers = True,
  footer_divider = True,
  index_col = "#",
  columns = df_passes_metrics_columns,
  even_row_color = row_colors["even"],
  footer_divider_kw = {"color": bg_color, "lw": 1},
  row_divider_kw = {"color": bg_color, "lw": 1},
  col_label_divider_kw = {"color": "#000000", "lw": 1},
  column_border_kw = {"color": "#000000", "lw": 1},
  textprops = {"fontsize": 13, "ha": "center"}
)

fig.text(0.04, 0.93, "Variação no Comportamento ao Nível do Passe Quando em Desvantagem", fontsize = 18, fontweight = "bold", ha = "left")
fig.text(0.04, 0.91, "Comparação da distância média e percentagem de passes completados com a classificação final", fontsize = 14, ha = "left", color = "#313131")
fig.text(0.04, 0.04, "DV>70': Indicadores quando a equipa se encontrava em desvantagem após os 70 minutos", fontsize = 13, ha = "left", color = "#313131");

Da tabela retiramos que __apenas duas equipas__ (Benfica e Nacional, de extremidades opostas da classificação) __têm uma distância média dos passes superior quando se encontram em desvantagem após os 70 minutos__, comparativamente à sua média global. De resto, todas as equipas procuram jogar mais curto quando se encontram nesta condição - ou são forçadas a isso pelos adversários em vantagem -, sendo as maiores diferenças (negativas) registadas para equipas de meio da tabela (Vitória, Famalicão, e Tondela). Desta forma, parece haver quase uma relação não linear em forma de parábola, com as __equipas do topo e do fundo da classificação a serem mais consistentes__ (com menor variação) __na distância média dos seus passes quando comparado com as equipas que terminaram a meio da tabela__.

Outra nota interessante: com exceção de dois outliers (Santa Clara, na primeira metade da tabela, e Rio Ave, na segunda), podemos observar uma __tendência para a distância média dos passes (como um todo) ir crescendo à medida que descemos na classificação final da liga.__ Este fator realça a importância de procurar um jogo associativo mais curto nesta liga, onde os blocos defensivos costumam estar muito bem organizados no último terço atacante e permitir menos espaço aos adversários (comparativamente com outras ligas de topo, como a Bundesliga ou até a própria La Liga e Premier League), mas onde é muitas vezes na zona intermédia do terreno que as equipas acabam por se superiorizar e criar situações para entrar no último terço com maior velocidade e perigo.

Relativamente à percentagem de passes completados, __a maior parte das equipas não parece demonstrar grande variação negativa quando se encontra em desvantagem após os 70 minutos face ao seu comportamento habitual__: apenas Vitória, Farense e Nacional (estes últimos, equipas que terminaram nos últimos lugares e desceram diretamente à Liga 2) têm uma percentagem de sucesso 2 pontos percentuais (pp) abaixo do seu valor global. Em sentido contrário, o campeão Sporting, Paços de Ferreira, Moreirense, Marítimo, B-SAD e Marítimo apresentam um aumento de acerto de pelo menos 3pp quando se encontram a perder nos últimos 20 minutos das partidas. Podemos argumentar, ainda assim, que __estes passes são consentidos pelos adversários que estão em vantagem__, que baixam o bloco e permitem a estas equipas jogar pela certa em zonas que não criam perigo de golo.

Ainda dentro da percentagem de passes completados quando em desvantagem após os 70 minutos, há novamente uma __tendência para este número decrescer à medida que descemos na classificação final da liga__ (sendo o Rio Ave, 16º, quase a única exceção clara), evidenciando a maior calma e segurança no passe por parte das equipas de topo, por contraste com os conjuntos da metade inferior da tabela.

Por fim, e seguindo o que poderíamos esperar à partida, há uma relação aparente entre a variação na distância média dos passes e a variação dos passes completados: __quando maior o aumento na procura pelo jogo curto, maior o aumento na percentagem de acerto__. Nota, ainda assim, para os casos do FC Porto, Vitória e Farense, que tentam jogar mais curto quando estão em situação de desvantagem nos últimos 20 minutos, mas ainda assim vêem a sua percentagem de passes completados diminuir (sendo que, no primeiro destes casos, a redução é apenas de 1pp).

<br><br>
Concluímos assim haver algumas variações interessantes no comportamento das equipas ao nível do passe quando se encontram em desvantagem no final das partidas, apesar de não ser clara e notória a existência de uma relação entre estas variações e a classificação no final do campeonato. Pese embora esta análise possa ser complementada com outros indicadores, estas métricas podem já ser úteis para as equipas técnicas conhecerem a abordagem dos adversários no final das partidas e poderem adaptar as suas estratégias defensivas (e potencialmente até de contra-ataque, se explorarmos as zonas do terreno que ficam mais descobertas nestes momentos das partidas).